<a href="https://colab.research.google.com/github/Nick97382000/Lettura_bilanci/blob/main/Lettura%20bilanci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lettura bilanci

Installo librerie di interesse e copio directory di riferimento da Git-Hub

In [39]:
!git clone https://github.com/Nick97382000/Lettura_bilanci
!pip install pdfplumber
!apt-get update -qq
!apt-get install -y tesseract-ocr tesseract-ocr-ita poppler-utils
!pip install pdf2image pytesseract pandas pillow

fatal: destination path 'Lettura_bilanci' already exists and is not an empty directory.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-ita is already the newest version (1:4.00~git30-7274cfa-1.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 15 not upgraded.


In [40]:
!git -C /content/Lettura_bilanci pull   # aggiorna i file
import importlib, funzioni_base, funz_sec_liv, config, ocr_func
importlib.reload(funzioni_base)
importlib.reload(funz_sec_liv)
importlib.reload(config)
importlib.reload(ocr_func)

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 5.87 KiB | 858.00 KiB/s, done.
From https://github.com/Nick97382000/Lettura_bilanci
   f4be250..b36e247  main       -> origin/main
Updating f4be250..b36e247
Fast-forward
 Lettura bilanci.ipynb | 337 ++++++++++++++++++++------------------------------
 config.py             | 118 ++++++++++++++++++
 ocr_func.py           | 256 ++++++++++++++++++++++++++++++++++++++
 3 files changed, 507 insertions(+), 204 deletions(-)
 create mode 100644 config.py
 create mode 100644 ocr_func.py


<module 'ocr_func' from '/content/Lettura_bilanci/ocr_func.py'>

###Definisco funzioni utili per le analisi delle stringhe

In [41]:
import sys
sys.path.insert(0, '/content/Lettura_bilanci')
from funzioni_base import *
from funz_sec_liv import *

Definisco funzioni per la lettura OCR dei documenti

Definisco funzione per leggere il pdf dalla cartella ed identificare la tabella OIC con i valori numerici di bilancio

In [44]:
def trova_tabella_bilancio(pdf_path, keywords):
    """
    trova e restituisce la tabella bilancio OIC, in argomento:
    - pdf_path: indica il percorso del file pdf da analizzare
    - keywords indica le parole chiave che identificano la prima pagina del bilancio
    """
    #leggo il pdf ed estraggo le tabelle; per ogni tabella la converto in un unica
    #stringa e controllo che le keywords siano presenti al suo interno; in caso
    #positivo salvo l'indice della pagina di riferimento.
    with pdfplumber.open(pdf_path) as pdf:
        pagina_start = None
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for ii, t in enumerate(tables):
                testo = " ".join(
                    str(cell).lower()
                    for row in t
                    for cell in row
                    if cell
                )
                if all(kw in testo for kw in keywords):
                    pagina_start = i
                    break;
            if pagina_start is not None:
                break

        if pagina_start is None:
            return None

        #legge a partire dalla tabella iniziale di riferimento, continua a leggere
        #finchè sono presenti tabelle nelle pagine successive, unendo tutto il
        #contenuto in un dataframe pandas
        tutte_le_righe = []
        intestazione = None

        for i in range (pagina_start, len(pdf.pages)):
                tables = pdf.pages[i].extract_tables()

                if not tables:
                    break

                for t in tables:
                    righe = [r for r in t if any(c for c in r)]
                    tutte_le_righe.extend(righe)

        if tutte_le_righe:
            n_colonne = max(len(r) for r in tutte_le_righe)
            tutte_le_righe = [r + [None]*(n_colonne - len(r)) for r in tutte_le_righe]

            df = pd.DataFrame(tutte_le_righe)
            return df
    return None


Definisco funzione che estrae i valori di interesse dalla tabella numerica OIC

In [45]:
import pandas as pd
import numpy as np

def summary_tabella(tabella, labels_dict):
    """
    analizza la tabella oic, restituendo i valori di interesse indicati in labels_dict
    """
    #gestisco in modo diverso se la tabella ha 5 colonne o 3, se ne ha 5 unisco
    #le prime due colonne
    if tabella.shape[1] == 3:
        date = tabella.iloc[0, 1:3].tolist()

    elif tabella.shape[1] == 5:
        tabella.iloc[:,0] = (
            tabella.iloc[:,0].fillna("").astype(str).str.strip()
                .str.cat(tabella.iloc[:,1].fillna("").astype(str).str.strip(),
                            sep=" ")
        )
        # elimina la colonna 1
        tabella.drop(tabella.columns[1], axis=1, inplace=True)

        # ricavo le date sui nuovi indici 1 e 2
        date = tabella.iloc[0, 1:3].tolist()

    else:
        return pd.DataFrame()

    col_label = tabella.columns[0]
    cols_valori = tabella.columns[1:3].tolist()

    righe = {}
    col_testi = tabella.iloc[:, col_label].astype(str)

    #scorre sui nomi da estrarre, applicando le logiche di include ed exclude
    for nome_output, regola in labels_dict.items():
        trovato = False

        for include in regola["include"]:
            mask = col_testi.apply(lambda x: match_testo(x, include, regola["exclude"]))
            if not mask.any():
                continue

            for _, row in tabella.loc[mask, cols_valori].iterrows():
                valori = row.tolist()
                if riga_valida(valori):
                    righe[nome_output] = [pulisci_numero(v) for v in valori]
                    trovato = True
                    break

            if trovato:
                break

        if not trovato:
            righe[nome_output] = [None] * len(cols_valori)

    righe["passivita_correnti"] = calcola_passivita_correnti(tabella, col_label, cols_valori,
                                                             iniz_deb, fin_deb, alias_correnti)

    return pd.DataFrame.from_dict(righe, orient="index", columns=date)



Leggo tutti i pdf nella cartella e stampo la tabella di sintesi

In [46]:
import glob
import os

pdf_files = glob.glob("Lettura_bilanci/deposito bilanci/*.pdf")

summary = {}

for pdf_path in pdf_files:
    nome = os.path.basename(pdf_path).replace(".pdf", "")
    tabella = trova_tabella_bilancio(pdf_path, keywords_first_page)

    if tabella is not None:
        summary[nome] = summary_tabella(tabella, campi_bilancio)



# stampa i risultati ottenuti
for nome, df in summary.items():
    print("\n" + "=" * 80)
    print(f"BILANCIO: {nome}")
    print("=" * 80)
    print(df)


BILANCIO: Volpato Industrie - Consolidato - 2024
                                           31-12-2024   31-12-2023
totale_attivo                              97737768.0  105606567.0
patrimonio_netto                           41957742.0   40978696.0
totale_attivo_circolante                   55074605.0   61397489.0
Totale_disponibilità_liquide               14156221.0   23944466.0
Totale_debiti                              50496961.0   59894867.0
Totale_rimanenze                           16435894.0   13791909.0
Totale_crediti_verso_clienti               19000152.0   17710304.0
Totale_debiti_verso_fornitori              16483302.0   15871700.0
Totale_debiti_verso_banche                 30375818.0   39636260.0
avviamento                                 16644208.0   16568852.0
ricavi_vendite_e_prestazioni               87375547.0   70278325.0
Utile_perdita_esercizio                     3956168.0    4658191.0
Risultato_prima_delle_imposte               6160080.0    5985159.0
Totale_ammor

Debug

In [47]:

pdf_path= "Lettura_bilanci/deposito bilanci/Conserve Italia - Civilistico e Consolidato - 2024.pdf"
#pdf_path= "Lettura_bilanci/deposito bilanci/Gollinucci - 2024.pdf"
tabella = trova_tabella_bilancio(pdf_path, keywords_first_page)
summary_tabella(tabella, campi_bilancio)


,30 giugno 2025,30 giugno 2024
totale_attivo,932834011.0,879915161.0
patrimonio_netto,305929558.0,300611685.0
totale_attivo_circolante,439099320.0,416207934.0
Totale_disponibilità_liquide,127609819.0,106847278.0
Totale_debiti,591475140.0,545013151.0
Totale_rimanenze,205612371.0,201996264.0
Totale_crediti_verso_clienti,55501954.0,60932295.0
Totale_debiti_verso_fornitori,236897353.0,228044458.0
Totale_debiti_verso_banche,191590518.0,168282068.0
avviamento,NaN,NaN


In [48]:
df = trova_tabella_oic_multipaagina(
    pdf_path="Lettura_bilanci/deposito bilanci/Kirey - 2024.pdf",
    start_marker="stato patrimoniale",
    end_marker="conto economico",
    lang="ita",
    dpi=300
)

df.head(50)
df.tail(50)

,label,data1,data2
76,Varie altre riserve,51,(2.879)
77,Totale altre riserve,64.138.921,17.230.521
78,VII Riserva per operazioni di copertura dei fl...,(381.859),381.860
79,-,None,None
80,VIII Utili (perdite) portati a nuovo,(16.360),(871.643)
81,-,None,None
82,IX Utile (perdita) dell'esercizio,(8.901.374),1.648.856
83,-,None,None
84,X Riserva negativa per azioni proprie in porta...,0,0
85,-,None,None


In [49]:
print(df)

                                                 label        data1  \
0                                               Attivo         None   
1    A) Crediti verso soci per versamenti ancora do...         None   
2    Totale crediti verso soci per versamenti ancor...         None   
3                                  B) Immobilizzazioni         None   
4                       | Immobilizzazioni immateriali         None   
..                                                 ...          ...   
121                                      Totale debiti   87.945.470   
122                                E) Ratei e risconti    8.575.634   
123                                     Totale passivo  160.159.167   
124                                              7 di.           73   
125                                                  -         None   

           data2  
0           None  
1           None  
2           None  
3           None  
4           None  
..           ...  
121   66.838.1